# Boreas / OpenPCDet — Kaggle T4 x2
GPU T4 x2, Internet ON; Add Input data; bật secret GITHUB_TOKEN. Token fine-grained: Resource owner danhyoyo, repo boreas, Contents Read-only. Tải archive qua GitHub API, không cần quyền ghi. Python 3.11 riêng bằng uv, kể cả kernel Python 3.13. PointPillar 128 beam, FP32, 80 epoch, batch 4/GPU (tổng 8), workers 4/GPU (tổng 8). VRAM không gộp; batch 4 chưa được xác nhận đủ VRAM trên Kaggle. Chạy thử EPOCHS=1 với TAG riêng trước. Notebook chưa thực thi trên Kaggle.

In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, json, tempfile, zipfile
import requests
from kaggle_secrets import UserSecretsClient

REPO_REF = "main"
DATA_ROOT = "/kaggle/input/datasets/danhyoyo/boreas-4-beams-level/boreas_openpcdet"
MODEL = "pointpillar"
BEAMS = 128
EPOCHS = 80
BATCH_PER_GPU = 4   # Tổng batch = 8 trên hai GPU.
WORKERS = 4         # Mỗi GPU: 4 worker; tổng 8.
SEED = 42
TAG = "kaggle_t4x2_pointpillar_beam128_seed42_fp32_bs8"
CKPT = ""
RESUME = False
REPO = Path("/kaggle/working/openPCDet")
ENV_DIR = Path("/kaggle/working/pcdet-py311")
PY = str(ENV_DIR / "bin/python")

def run(args, cwd=None, env=None):
    child_env = os.environ.copy()
    if env is not None:
        child_env.update(env)
    child_env["PYTHONUNBUFFERED"] = "1"
    print("Running:", " ".join(map(str, args)), flush=True)
    with subprocess.Popen(list(map(str, args)), cwd=cwd, env=child_env,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, errors="replace", bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, args)

run(["nvidia-smi"])
assert not (CKPT and RESUME)
print("Cấu hình sẵn sàng. Batch tổng:", 2 * BATCH_PER_GPU,
      "| workers tổng:", 2 * WORKERS, flush=True)


In [ ]:
import time, threading
from contextlib import contextmanager

@contextmanager
def stage(label):
    print(label, flush=True)
    started = time.monotonic()
    stop = threading.Event()
    def heartbeat():
        while not stop.wait(15):
            print(f"{label}: vẫn chạy, {time.monotonic()-started:.0f}s", flush=True)
    thread = threading.Thread(target=heartbeat, daemon=True)
    thread.start()
    try:
        yield
    finally:
        stop.set()
        thread.join()
        print(f"Kết thúc bước: {label} ({time.monotonic()-started:.1f}s)", flush=True)

def download_code():
    if (REPO / "tools/boreas_benchmark/beam_config.py").is_file():
        print("Code đã có, bỏ qua tải.", flush=True)
        return
    if REPO.exists():
        raise RuntimeError("Thư mục code không đầy đủ. Dùng phiên mới hoặc đổi REPO sang thư mục mới.")
    headers = {}
    try:
        with stage("1/4 Đọc Kaggle secret"):
            headers.update(Authorization="Bearer " + UserSecretsClient().get_secret("GITHUB_TOKEN").strip(),
                           Accept="application/vnd.github+json")
        with stage("2/4 Kiểm tra quyền đọc repo"):
            with requests.get(
                "https://api.github.com/repos/danhyoyo/boreas/contents/setup.py",
                params={"ref": REPO_REF}, headers=headers, timeout=(15, 30)) as response:
                if response.status_code != 200:
                    raise RuntimeError(
                        f"GitHub HTTP {response.status_code}: token cần Resource owner=danhyoyo, "
                        "repo boreas, Contents Read-only; kiểm tra hạn token và REPO_REF.")
        with tempfile.TemporaryDirectory(dir="/kaggle/working") as temp_dir:
            archive = Path(temp_dir) / "code.zip"
            with stage("3/4 Tải ZIP code"):
                started = time.monotonic()
                downloaded = 0
                last_report = started
                with requests.get(
                    f"https://api.github.com/repos/danhyoyo/boreas/zipball/{REPO_REF}",
                    headers=headers, stream=True, timeout=(15, 30)) as response:
                    if response.status_code != 200:
                        raise RuntimeError(f"Archive HTTP {response.status_code}")
                    total = int(response.headers.get("Content-Length", 0))
                    print("Đã nhận kết nối tải ZIP.", flush=True)
                    with archive.open("wb") as output:
                        for chunk in response.iter_content(256 * 1024):
                            now = time.monotonic()
                            if now - started > 600:
                                raise TimeoutError("Tải ZIP quá 10 phút; kiểm tra repo có data/checkpoint lớn.")
                            if chunk:
                                output.write(chunk)
                                downloaded += len(chunk)
                            if now - last_report >= 5:
                                suffix = f"/{total/1024**2:.1f} MiB" if total else ""
                                print(f"Đã tải {downloaded/1024**2:.1f} MiB{suffix}; "
                                      f"{downloaded/1024**2/max(now-started,1):.2f} MiB/s", flush=True)
                                last_report = now
                    print(f"ZIP hoàn tất: {downloaded/1024**2:.1f} MiB", flush=True)
            with stage("4/4 Kiểm tra và giải nén"):
                unpack = Path(temp_dir) / "unpack"
                unpack.mkdir()
                with zipfile.ZipFile(archive) as z:
                    members = z.infolist()
                    print(f"{len(members)} mục; giải nén {sum(m.file_size for m in members)/1024**2:.1f} MiB",
                          flush=True)
                    for member in members:
                        if not (unpack / member.filename).resolve().is_relative_to(unpack.resolve()):
                            raise RuntimeError("Archive có đường dẫn không hợp lệ")
                    for i, member in enumerate(members, 1):
                        z.extract(member, unpack)
                        if i % 500 == 0:
                            print(f"Giải nén {i}/{len(members)} mục", flush=True)
                roots = list(unpack.iterdir())
                assert len(roots) == 1 and (roots[0] / "setup.py").is_file()
                assert (roots[0] / "tools/boreas_benchmark/beam_config.py").is_file()
                revision = roots[0].name
                shutil.move(str(roots[0]), str(REPO))
                (REPO / "kaggle_source_revision.txt").write_text(revision)
    finally:
        headers.clear()

download_code()
print("Code sẵn sàng:", REPO, flush=True)


In [ ]:
# Argoverse là dependency tùy chọn; chỉ import khi chọn Argo2Dataset.
# Boreas dùng CustomDataset, nên không cần cài SDK av2.
import ast
dataset_init = REPO / "pcdet/datasets/__init__.py"
source = dataset_init.read_text()
marker = "# Kaggle: load Argoverse SDK only when its dataset is selected."
if marker not in source:
    import_line = "from .argo2.argo2_dataset import Argo2Dataset"
    registry_line = "    'Argo2Dataset': Argo2Dataset"
    needle = "    dataset = __all__[dataset_cfg.DATASET]("
    assert source.count(import_line) == 1, "Dataset registry khác bản dự kiến; cần kiểm tra code."
    assert source.count(registry_line) == 1 and source.count(needle) == 1
    updated = source.replace(import_line, marker).replace(registry_line, "")
    lazy_import = (
        '    if dataset_cfg.DATASET == "Argo2Dataset":\n'
        '        from .argo2.argo2_dataset import Argo2Dataset\n'
        '        __all__["Argo2Dataset"] = Argo2Dataset\n\n'
    )
    updated = updated.replace(needle, lazy_import + needle)
    ast.parse(updated)
    backup = dataset_init.with_suffix(".py.before_kaggle_lazy_import")
    if not backup.exists():
        backup.write_text(source)
    dataset_init.write_text(updated)
print("Dataset registry: Boreas không còn import av2 khi khởi động.", flush=True)


In [ ]:
# Kernel Kaggle có thể là Python 3.13; mọi lệnh build/train dùng Python 3.11 riêng.
os.environ["UV_PYTHON_INSTALL_DIR"] = "/kaggle/working/uv-python"
os.environ["UV_CACHE_DIR"] = "/kaggle/working/uv-cache"
run([sys.executable, "-m", "pip", "install", "uv"])
run([sys.executable, "-m", "uv", "python", "install", "3.11"])
if not Path(PY).exists():
    run([sys.executable, "-m", "uv", "venv", "--python", "3.11",
         "--managed-python", "--seed", ENV_DIR])
run([PY, "-c", "import sys; print(sys.version); assert sys.version_info[:2] == (3,11)"])

nvcc = shutil.which("nvcc")
assert nvcc, "Cần CUDA toolkit/nvcc để build extension."
v = subprocess.check_output([nvcc, "--version"], text=True)
print(v)
assert "release 12." in v, "Baseline cần CUDA toolkit 12.x; gửi log nvcc nếu image khác."
os.environ.update(CUDA_HOME=str(Path(nvcc).resolve().parents[1]),
                  TORCH_CUDA_ARCH_LIST="7.5", MAX_JOBS="2")
run([PY, "-m", "pip", "install", "pip", "setuptools==72.1.0", "wheel", "ninja"])
run([PY, "-m", "pip", "install", "torch==2.5.1", "torchvision==0.20.1",
     "--index-url", "https://download.pytorch.org/whl/cu121"])
run([PY, "-m", "pip", "install", "numpy==1.26.4", "numba==0.60.0",
     "spconv-cu120==2.3.6", "tensorboardX", "easydict", "pyyaml",
     "scikit-image", "tqdm", "SharedArray", "opencv-python-headless==4.10.0.84",
     "pyquaternion", "pandas", "scipy", "av", "kornia"])
run([PY, "setup.py", "build_ext", "--inplace"], cwd=REPO)
run([PY, "-m", "pip", "install", "-e", ".", "--no-build-isolation", "--no-deps"], cwd=REPO)
run([PY, "-m", "pip", "freeze"])


In [ ]:
assert BEAMS in (128,64,32,16)
target=Path(DATA_ROOT)/str(BEAMS)
for name in ["manifest.json","custom_infos_train.pkl","custom_infos_val.pkl","ImageSets/train.txt","ImageSets/val.txt","points","labels"]:
    assert (target/name).exists(),str(target/name)
m=json.loads((target/"manifest.json").read_text())
assert m.get("beam_count")==BEAMS and not m.get("smoke_test_only")
for split in ("train","val"):
    ids=(target/f"ImageSets/{split}.txt").read_text().split()
    assert ids and (target/"points"/f"{ids[0]}.npy").is_file()
    print(split,len(ids))
run([PY,"-c","import torch; import spconv.pytorch; from pcdet.models import build_network; from pcdet.ops.iou3d_nms import iou3d_nms_cuda; assert torch.cuda.device_count()==2; print(torch.__version__,torch.version.cuda); print([torch.cuda.get_device_name(i) for i in range(2)])"],cwd=REPO)


In [ ]:
check=REPO/"kaggle_ddp_check.py"
check.write_text("""import os,torch
import torch.distributed as d
from datetime import timedelta
r=int(os.environ['LOCAL_RANK'])
torch.cuda.set_device(r)
d.init_process_group('nccl',timeout=timedelta(seconds=120))
x=torch.ones(1,device='cuda')
d.all_reduce(x)
assert x.item()==2
print('NCCL OK GPU',r,flush=True)
d.destroy_process_group()
""")
run([PY,"-m","torch.distributed.run","--standalone","--nnodes=1","--nproc_per_node=2",check],cwd=REPO)


In [ ]:
assert EPOCHS>=1 and BATCH_PER_GPU>=1
assert (REPO/f"tools/cfgs/boreas_models/{MODEL}.yaml").is_file()
if CKPT: assert Path(CKPT).is_file()
env=os.environ.copy()
env.update(CUDA_VISIBLE_DEVICES="0,1",OMP_NUM_THREADS="1",NCCL_DEBUG="WARN")
args=[PY,"-m","torch.distributed.run","--standalone","--nnodes=1","--nproc_per_node=2","train.py","--launcher","pytorch","--cfg_file",f"cfgs/boreas_models/{MODEL}.yaml","--batch_size",str(2*BATCH_PER_GPU),"--epochs",str(EPOCHS),"--workers",str(WORKERS),"--fix_random_seed","--seed",str(SEED),"--no_amp","--boreas_beams",str(BEAMS),"--extra_tag",TAG,"--ckpt_save_interval","1","--max_ckpt_save_num","3","--range_eval"]
if CKPT: args+=["--ckpt",CKPT]
elif RESUME: args+=["--resume"]
args+=["--set","DATA_CONFIG.DATA_PATH",str(target)]
run(args,cwd=REPO/"tools",env=env)


In [ ]:
for p in sorted((REPO/"output").rglob("*.pth")):
    print(p,round(p.stat().st_size/1024**2,1),"MiB")
for p in sorted((REPO/"output").rglob("accuracy.json")):
    print(p)
    print(p.read_text()[:3000])


Save Version với output và kiểm tra checkpoint trong /kaggle/working/openPCDet/output. Phiên sau Add Input checkpoint rồi đặt CKPT; EPOCHS là tổng epoch mục tiêu; giữ model/beam/seed/batch/precision. RESUME chỉ tìm output hiện tại. Benchmark mới: đổi TAG, CKPT rỗng. MODEL: pointpillar, pointpillar_lite, second, centerpoint, pv_rcnn, voxelnext. FP32 mặc định phù hợp PV-RCNN hiện tại. Nếu OOM batch1/GPU, cần model nhẹ hơn hoặc sửa representation có ghi nhận benchmark. Nguồn: https://docs.pytorch.org/docs/stable/elastic/run.html ; https://github.com/traveller59/spconv

## Metrics trên bốn beam
Chạy sau train hoặc đặt EVAL_CKPT tới checkpoint đã lưu. Cùng checkpoint đánh giá 128/64/32/16 beam: test-time robustness, không train lại từng beam. AP40 local (không phải official Boreas); Precision/Recall/F1 ở score 0.1. Profiling batch1 FP32 một T4, 200 frames và 20 warmup, không chạy đồng thời train hoặc phiên GPU khác. EVAL_BATCH=4, EVAL_WORKERS=4 độc lập với batch train 4/GPU. Kết quả CSV/JSON trong working/output.

In [ ]:
# Đánh giá CÙNG checkpoint trên 128/64/32/16 beam (test-time robustness).
import re, csv
METRIC_BEAMS = (128, 64, 32, 16)
EVAL_CKPT = ""          # Điền checkpoint khác nếu cần; mặc định epoch EPOCHS của lượt train.
EVAL_BATCH = 4          # Inference một GPU; không phải batch train.
EVAL_WORKERS = 4
PROFILE_EACH_BEAM = True
PROFILE_FRAMES = 200
PROFILE_WARMUP = 20

suffix = f"_beam{BEAMS}"
train_tag = TAG if TAG.endswith(suffix) else TAG + suffix
train_dir = REPO / "output/boreas_models" / MODEL / train_tag
metric_ckpt = Path(EVAL_CKPT) if EVAL_CKPT else train_dir / "ckpt" / f"checkpoint_epoch_{EPOCHS}.pth"
assert metric_ckpt.is_file(), f"Thiếu checkpoint: {metric_ckpt}; đặt EVAL_CKPT đúng file."
numbers = re.findall(r"\d+", str(metric_ckpt))
epoch_id = numbers[-1] if numbers else "no_number"
eval_tag = train_tag + "_fourbeam_metrics"
eval_base = REPO / "output/boreas_models" / MODEL / eval_tag
metric_env = os.environ.copy()
metric_env.update(CUDA_VISIBLE_DEVICES="0", OMP_NUM_THREADS="1")
# Profiling không được đồng thời với train/job GPU khác.
assert metric_env.get("CUDA_LAUNCH_BLOCKING") != "1", "Tắt CUDA_LAUNCH_BLOCKING trước profiling."

baseline_ids = (Path(DATA_ROOT) / "128/ImageSets/val.txt").read_text().split()
for beam in METRIC_BEAMS:
    data = Path(DATA_ROOT) / str(beam)
    for name in ("manifest.json", "custom_infos_val.pkl", "ImageSets/val.txt"):
        assert (data / name).is_file(), f"Thiếu {data/name}"
    manifest = json.loads((data / "manifest.json").read_text())
    assert manifest.get("beam_count") == beam and not manifest.get("smoke_test_only")
    assert (data / "ImageSets/val.txt").read_text().split() == baseline_ids, (
        f"Validation IDs beam {beam} khác baseline 128.")
print("Checkpoint:", metric_ckpt, "| validation:", len(baseline_ids), flush=True)

beam_results = {}
for beam in METRIC_BEAMS:
    print(f"===== EVALUATE {beam} BEAMS =====", flush=True)
    run([PY, "test.py", "--cfg_file", f"cfgs/boreas_models/{MODEL}.yaml",
         "--ckpt", metric_ckpt, "--batch_size", str(EVAL_BATCH),
         "--workers", str(EVAL_WORKERS), "--seed", str(SEED),
         "--extra_tag", eval_tag, "--eval_tag", f"beam{beam}", "--range_eval",
         "--set", "DATA_CONFIG.DATA_PATH", str(Path(DATA_ROOT) / str(beam))],
        cwd=REPO / "tools", env=metric_env)
    result_dir = eval_base / "eval" / f"epoch_{epoch_id}" / "val" / f"beam{beam}"
    assert (result_dir / "range_ap.json").is_file(), f"Thiếu metric: {result_dir}"
    beam_results[beam] = result_dir
    ap = json.loads((result_dir / "range_ap.json").read_text())
    print(f"Beam {beam}: local mAP40 = {ap['ranges']['all']['mAP40']}", flush=True)


In [ ]:
# Batch-1 FP32 trên MỘT T4: latency gồm full forward + NMS và end-to-end.
# Ghi config riêng để profiler không dùng DATA_PATH của máy local.
if PROFILE_EACH_BEAM:
    for beam, result_dir in beam_results.items():
        profile_cfg = REPO / "tools/cfgs/kaggle_profile" / f"{MODEL}_beam{beam}.yaml"
        profile_cfg.parent.mkdir(parents=True, exist_ok=True)
        make_cfg = """
import sys,yaml
from pathlib import Path
source,dest,data=sys.argv[1:]
config=yaml.safe_load(Path(source).read_text())
config["DATA_CONFIG"]["DATA_PATH"]=data
Path(dest).write_text(yaml.safe_dump(config,sort_keys=False))
"""
        run([PY, "-c", make_cfg,
             REPO / f"tools/cfgs/boreas_models/{MODEL}.yaml",
             profile_cfg, Path(DATA_ROOT) / str(beam)], cwd=REPO / "tools")
        print(f"===== PROFILE {beam} BEAMS =====", flush=True)
        run([PY, "boreas_benchmark/profile.py", "--cfg", profile_cfg,
             "--ckpt", metric_ckpt, "--boreas_beams", str(beam),
             "--seed", str(SEED), "--frames", str(PROFILE_FRAMES),
             "--warmup", str(PROFILE_WARMUP), "--out", result_dir / "profile.json"],
            cwd=REPO / "tools", env=metric_env)


In [ ]:
# Tổng hợp AP, range AP, Precision/Recall/F1 và efficiency thành CSV + JSON.
rows = []
details = {}
for beam, result_dir in beam_results.items():
    ap = json.loads((result_dir / "range_ap.json").read_text())
    assert ap.get("protocol_version") == "local_ap40_v1"
    row = {"model": MODEL, "train_beams": BEAMS, "test_beams": beam,
           "checkpoint": str(metric_ckpt), "mAP40": ap["ranges"]["all"]["mAP40"]}
    for range_name, values in ap["ranges"].items():
        row[f"mAP40_{range_name}"] = values["mAP40"]
        for class_name, metrics in values["classes"].items():
            row[f"{class_name}_AP40_{range_name}"] = metrics["AP40"]
            if range_name == "all":
                for key, value in metrics["operating_point"].items():
                    row[f"{class_name}_{key}"] = value
    profile_path = result_dir / "profile.json"
    timing = json.loads(profile_path.read_text()) if profile_path.exists() else None
    if timing:
        assert timing["precision"] == "FP32" and timing["batch_size"] == 1
        row.update(FPS=timing["fps_from_mean_end_to_end"],
                   latency_mean_ms=timing["end_to_end"]["mean_ms"],
                   latency_p50_ms=timing["end_to_end"]["p50_ms"],
                   latency_p95_ms=timing["end_to_end"]["p95_ms"],
                   forward_with_NMS_mean_ms=timing["model_forward_including_postprocess"]["mean_ms"],
                   peak_GPU_MiB=timing["peak_cuda_allocated_mib"],
                   parameters=timing["parameters"])
    rows.append(row)
    details[str(beam)] = {"accuracy": ap, "profile": timing}
baseline_ap = next(row["mAP40"] for row in rows if row["test_beams"] == 128)
for row in rows:
    row["drop_vs_128_pp"] = (baseline_ap - row["mAP40"]
                             if baseline_ap is not None and row["mAP40"] is not None else None)

summary_dir = eval_base / f"fourbeam_epoch_{epoch_id}"
summary_dir.mkdir(parents=True, exist_ok=True)
fields = list(dict.fromkeys(key for row in rows for key in row))
with (summary_dir / "beam_metrics.csv").open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=fields)
    writer.writeheader()
    writer.writerows(rows)
(summary_dir / "beam_metrics.json").write_text(json.dumps(details, indent=2))
from IPython.display import display, Markdown
columns = ["test_beams", "mAP40", "drop_vs_128_pp", "FPS", "latency_mean_ms", "peak_GPU_MiB"]
table = "| " + " | ".join(columns) + " |\n"
table += "| " + " | ".join("---" for _ in columns) + " |\n"
for row in rows:
    table += "| " + " | ".join(str(row.get(key, "")) for key in columns) + " |\n"
display(Markdown(table))
print("Kết quả:", summary_dir, flush=True)
